# ASISTENTE VACACIONAL CON RAG Y TOOLS

## Imports, definición de modelos, creación de cliente

Cargamos la API key de Google en `GOOGLE_API_KEY`

In [22]:
import os, getpass
from dotenv import load_dotenv
from google import genai
from pathlib import Path
import pymupdf
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
import faiss
from google.genai import types
from typing import Any


# cargamos variables desde .env 
dotenv_loaded = load_dotenv()

# solicita claves si no están definidas
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Introduce la API de Google: ")

# error si después de todo no tenemos API KEYs
if not os.getenv("GOOGLE_API_KEY"):
    raise RuntimeError("Claves no encontradas.")

print("Claves cargadas.")

Claves cargadas.


In [23]:
GENERATION_MODEL = os.getenv("GOOGLE_GENERATION_MODEL", "gemini-3.5-flash-lite")
EMBEDDING_MODEL = os.getenv("GOOGLE_EMBEDDING_MODEL", "gemini-embedding-001")

print("Modelo de generación: ", GENERATION_MODEL)
print("Modelo de embeddings: ", EMBEDDING_MODEL)

Modelo de generación:  gemini-3.5-flash-lite"
Modelo de embeddings:  gemini-embedding-001


In [24]:
client = genai.Client()

## Carga PDF RAG

In [25]:
PDF_PATH = Path("data/TENERIFE.pdf")

pages = []

# leemos cada página del pdf
with pymupdf.open(PDF_PATH) as doc:
    # page representa el contenido de la propia hoja, i representa el número de página
    for i, page in enumerate(doc, start=1):
        pages.append(
            {
                "page": i,
                "content": page.get_text()
            }
        )

print("Número de páginas cargadas:", len(pages))
print("Primeros caracteres:\n")
print(pages[0]["content"][:300])


Número de páginas cargadas: 25
Primeros caracteres:

TENERIFE – LUGARES DE INTERÉS 
SITIOS QUE VER 
 
ZONA NORTE 
 
• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- 
Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- 
Caminar por la Avenida Marítima hasta Pl


## Chunking

In [26]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", ", ", " ", ""]
)

chunks = []

for page in pages:
    page_chunks = splitter.split_text(page["content"])

    for chunk in page_chunks:
        chunks.append(
            {
                "chunk_id": len(chunks),
                "page": page["page"],
                "text": chunk
            }
        )

print(f"Se han obtenido {len(chunks)} chunks.")
print()
print("Chunk 1:")
print(chunks[0])

Se han obtenido 37 chunks.

Chunk 1:
{'chunk_id': 0, 'page': 1, 'text': 'TENERIFE – LUGARES DE INTERÉS \nSITIOS QUE VER \n \nZONA NORTE \n \n• Santa Cruz de Tenerife: \nSanta Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa \nCruz sería: \n- \nAparcar en el aparcamiento del Parque Marítimo (ubicación). \n- \nCaminar por la Avenida Marítima hasta Plaza de España (ubicación). \n- \nPor el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). \n- \nUna vez llegados a Plaza España, callejear un poco (subir la Calle Castillo \ndirección Plaza Weyler - ubicación –; ir hacia el Parque García Sanabria - \nubicación -; y bajar de nuevo hacia Plaza de España pasando por la Plaza del \nPríncipe - ubicación). \n-'}


Para el proceso de chunking se utliza la función recursiva de langchain `RecursiveCharacterTextSplitter`. Se ha elegido porque se ha considerado la opción más óptima de hacer el chunking. Sigue una jerarquía de separadores para dividir los chunks, ordenados de mayor a menor importancia, estipulada en el parámetro `separators`.

## Embedding 

In [27]:
def embed_texts(texts: list[str], task_type: str) -> np.array:
    """Embed lista y devuelve matriz numpy. Utilizaremos dos tipos de tarea: 
        - RETRIEVAL_DOCUMENT: para los chunks del PDF.
        - RETRIEVAL_QUERY: para las preguntas del usuario."""
    # llamamos al modelo de embedding utilizando el cliente
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(
            task_type=task_type
        )
    )
    # guardamos vectores de embeddings 
    vectors = [item.values for item in response.embeddings]
    return np.array(vectors, dtype=np.float32)


In [28]:
# guardamos solo texto de todos los chunks
chunk_texts = [chunk["text"] for chunk in chunks]

# Llamamos a función de embedding
chunk_embeddings = embed_texts(
    texts=[chunk_texts],
    task_type="RETRIEVAL_DOCUMENT"
)


# normalizamos vectores para similitud coseno
faiss.normalize_L2(chunk_embeddings)

# obtenemos dimensiones de ebeddings
dimension = chunk_embeddings.shape[1]

# creamos indice FAISS
index = faiss.IndexFlatIP(dimension)

# añadimos embeddings
index.add(chunk_embeddings)

print(f"Vectores almacenados: {index.ntotal}")


Vectores almacenados: 1


`faiss.normalize_L2`: convierte los embeddings en vectores de longitud 1.

`IndexFlatIP`: crea un índice que utiliza el producto interno y realiza una búsqueda exacta.

`index.add`: incorpora los embeddings al índice.

## Recuperación del contexto

In [29]:
def search_internal_docs(query: str, index, k: int = 4) -> list[dict[str, Any]]:
    """Recupera k chunks más relevantes"""
    # embedding de la query
    query_embedding = embed_texts(
        texts=[query],
        task_type="RETRIEVAL_QUERY"
    )

    # normalizamos 
    faiss.normalize_L2(query_embedding)

    # buscamos k vectores similares 
    scores, indexes = index.search(
        x=query_embedding,
        k=min(k, index.ntotal)
    )

    results = []
    # index.search() devuelve dos matrices, puede recibir más de una pregunta en la misma llamada, en [0] está la primera pregunta
    for rank, (score, idx) in enumerate (zip(scores[0], indexes[0]), start=1):
        chunk = chunks[int(idx)]
        results.append(
            {
                "rank": rank,
                "score": float(score),
                "chunk_id": chunk["chunk_id"],
                "page": chunk["page"],
                "text": chunk["text"]
            }
        )

    return results

def print_recovered_chunks(results) -> None:
    for result in results:
        print("Chunk rank: ", result["rank"])
        print("Score: ", result["score"])
        print("chunk_id: ", result["chunk_id"])
        print("Page: ", result["page"])
        print("Content:")
        print(result["text"])
        print()
        print()

In [35]:
# prueba de función de recuperación de contexto 
results_1 = search_internal_docs(
    query="¿Qué miradores se recomiendan para visitar el Parque Rural de Anaga?",
    index=index,
    k=3
)

print_recovered_chunks(results_1)

Chunk rank:  1
Score:  0.7613589763641357
chunk_id:  6
Page:  5
Content:
la Playa de Benijo (imprescindible que esté soleado para poder ver el atardecer en 
esta playa, ya que es espectacular). 
 
Algunos de los miradores que os recomiendo en la ruta por Anaga: 
o Mirador de La Jardina (diría que es mi mirador favorito) [ubicación]


Chunk rank:  2
Score:  0.7611202001571655
chunk_id:  5
Page:  5
Content:
• Parque rural de Anaga 
Es un espacio natural protegido, declarado Reserva de la Biosfera y que cuenta con la 
mayor cantidad de endemismos de Europa. 
Tiene bastantes miradores chulísimos y rutas que podéis hacer si os mola este tipo 
de planes y si teníais pensado alguna caminata (véase Las Mejores Rutas en Parque 
Rural de Anaga). 
Visitar esta zona de la isla es un must y lo suyo sería que aprovecharais un día, os 
levantarais temprano, visitarais La Laguna y desayunarais por allí y ya luego rollo 
11:00-12:00 tiréis hacia Anaga, parando en algunos de los miradores y llegando has

In [40]:
results_2 = search_internal_docs(
    query="¿Dónde puedo disfrutar de la gastronomía tradicional canaria en un restaurante auténtico?",
    index=index,
    k=3
)

print_recovered_chunks(results_2)

Chunk rank:  1
Score:  0.6999064087867737
chunk_id:  34
Page:  24
Content:
[Tripadvisor] (os podría recomendar más guachinches, pero diría que 
este es mi favorito) 
 
Nota: los Guachinches son típicos en la zona norte de Tenerife. 
Actualmente existen muchos restaurantes que se llaman Guachinches 
pero no son los originales, ya que tienen una carta mucho más amplia 
y, por así decirlo, es más como un restaurante común (aunque la 
comida también estará increíble), pero la experiencia de Guachinche 
Guachinche es un sitio como el que he indicado.  
 
▪ Pizzería Gioconda (zona de La Orotava [ubicación]) [Tripadvisor] 
▪ El Palestra (zona de La Orotava [ubicación]. Este es el típico sitio que 
ha estado toda la vida en La Orotava y que te pides a domicilio cuando


Chunk rank:  2
Score:  0.694322943687439
chunk_id:  33
Page:  24
Content:
▪ Guachinche El Cubano (zona de Santa Úrsula [ubicación]. Para llegar 
atravesaréis una especie de camino de tierra, meteos sin miedo. Si 
vais a Tenerif

In [30]:
# función para formatear contexto guardando fuente
def format_results_with_sources(results) -> str:
    blocks = []
    for i, res in enumerate(results, start=1):
        score = res.get("score")
        page = res.get("page", "?")
        chunk_id = res.get("chunk_id", "?")
        text = res.get("text")
        blocks.append(
            f"Score {score:.3f} Página {page}, chunk {chunk_id}\n{text}"
        )
    return "\n\n".join(blocks)

## Log para llamadas a herramientas 

In [31]:
tool_logs = []

def register_tool_call(name, arguments, result, error = None) -> None:
    tool_logs.append({
        "tool": name,
        "arguments": arguments,
        "success": error is None,
        "result": result,
        "error": str(error) if error else None
    })

## RAG como herramienta 

In [32]:
def search_internal_docs_tool(query: str, k: int = 3) -> str:
    """Busca información turística en la guía de Tenerife y devuelve los k fragmentos más relevantes con sus páginas."""
    try:
        results = search_internal_docs(query, index, k)
        register_tool_call(
            name="search_internal_docs_tool",
            arguments={
                "query": query,
                "k": k
            },
            result=results
        )
        return format_results_with_sources(results)
    except Exception as exc:
        register_tool_call(
            name="search_internal_docs_tool",
            arguments={
                "query": query,
                "k": k
            },
            result=None,
            error=exc
        )
        return f"ERROR: {str(exc)}"

## Instrucciones para los distintos modelos

In [33]:
MAIN_LLM_SYSTEM_INSTRUCTIONS = """
    Eres un asistente turístico especializado en Tenerife.
    Tu objetivo es ayudar al usuario a organizar su viaje, recomendar lugares
    de interés y resolver dudas turísticas de manera clara, útil y cercana.
    
    USO DE HERRAMIENTAS:
    - Dispones de herramientas para consultar una guía turística de Tenerife
      y obtener predicciones meteorológicas.
    - Utiliza search_internal_docs_tool cuando el usuario pregunte por playas,
      monumentos, restaurantes, actividades, rutas o recomendaciones turísticas.
    - Basa tus recomendaciones en la información recuperada de la guía.
    - Utiliza get_weather cuando el usuario pregunte por el tiempo, las
      temperaturas, la lluvia o las condiciones meteorológicas de una fecha.
    - Si una recomendación depende del tiempo, utiliza ambas herramientas
      para combinar la información turística con la meteorológica.
    - No inventes datos meteorológicos ni afirmes haber consultado una
      herramienta si no la has ejecutado.
    
    GESTIÓN DE INFORMACIÓN:
    - Prioriza la información obtenida de las herramientas.
    - Cita las páginas de la guía turística utilizadas en tus respuestas.
    - Si la guía no contiene la información solicitada, indícalo claramente.
    - Si una herramienta devuelve un error, explícalo sin inventar resultados.
    - No presentes como actuales datos que puedan haber cambiado, como
      horarios, precios, restricciones de acceso o disponibilidad.
    
    CONTEXTO CONVERSACIONAL:
    - Ten en cuenta el historial proporcionado para mantener la coherencia.
    - Recuerda las preferencias, fechas y lugares mencionados anteriormente.
    - Interpreta preguntas de seguimiento teniendo en cuenta la conversación.
    - No vuelvas a consultar herramientas innecesariamente si la información
      anterior sigue siendo suficiente y válida.
    
    ESTILO DE RESPUESTA:
    - Responde siempre en español.
    - Utiliza un lenguaje natural, cercano y fácil de entender.
    - Ofrece respuestas concretas, evitando explicaciones innecesarias.
    - Organiza las recomendaciones de forma clara cuando sea útil.
    - Si no dispones de información suficiente, reconócelo.
"""

SUMMARIZE_LLM_SYSTEM_INSTRUCTIONS = """
    Eres un asistente especializado en resumir historiales de conversación.
    Tu tarea es condensar la conversación proporcionada para que otro
    asistente pueda continuarla sin perder información relevante.
    Instrucciones:
    - Conserva las preferencias, necesidades y restricciones del usuario.
    - Conserva los lugares turísticos, fechas y actividades mencionados.
    - Conserva las recomendaciones realizadas y las decisiones tomadas.
    - Mantén los datos necesarios para comprender futuras preguntas.
    - Elimina saludos, repeticiones e información irrelevante.
    - No inventes información ni añadas conocimientos externos.
    - No incluyas información que no esté presente en el historial.
    - Redacta un resumen breve, claro y estructurado.                    
    - Devuelve únicamente el texto del resumen, sin explicaciones adicionales.
"""

## Control de historial

In [34]:
MAX_TOKENS = 5000    

def exceeds_token_limit(conversation: list[types.Content]) -> bool:
    """True si se ha sobrepasado el límite de tokens establecido"""
    token_count = client.models.count_tokens(
        model=GENERATION_MODEL,
        contents=conversation
    )
    if token_count.total_tokens > MAX_TOKENS:
        return True
    else:
        return False

def conversational_context_control(conversation: list[types.Content], query: str = None) -> list[types.Content]:
    """Resume el historial si sobrepasa el límite de tokens, devuelve historial resumido"""
    if exceeds_token_limit(conversation):
        # llamada a modelo para que resuma el historial
        summary = client.models.generate_content(
            model=GENERATION_MODEL,
            contents=conversation[:-1], # excluimos última pregunta del user
            config=types.GenerateContentConfig(
                system_instruction=SUMMARIZE_LLM_SYSTEM_INSTRUCTIONS
            )
        )
        conversation: list[types.Content] = [] 
        conversation.extend(
            [
                types.Content(
                    role = "user",
                    parts=[types.Part.from_text(text=f"Resumen de la conversación anterior:\n{summary.text}")]
                ),
                types.Content(
                        role = "user",
                        parts=[types.Part.from_text(text=query)]
                )
            ]
        )
    return conversation

    

In [35]:
def run_llm_with_tools(user_input: str, conversation: list[types.Content], max_tool_calls: int = 5) -> list[dict[str, Any]]:
    """Invoca al modelo con herramientas; devulve respuesta generada, herramientas utilizadas y contexto"""
    
    # control del historial y tokens
    conversation.append(
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=user_input)]
        )
    )
    conversation_and_query = conversational_context_control(conversation, user_input)
    
    # llamada al cliente con tools y ejecución automática de las mismas
    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=conversation_and_query,
        config=types.GenerateContentConfig(
            system_instruction=MAIN_LLM_SYSTEM_INSTRUCTIONS,
            tools=[search_internal_docs_tool],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(
                maximum_remote_calls=max_tool_calls
            )
        )
    )

    # añadimos la respuesta
    conversation_and_query.append(
        types.Content(
            role="model",
            parts=[types.Part.from_text(text=response.text)]
        )
    )
    
    return {
        "text": response.text,
        "conversation": conversation_and_query
    }
    

In [ ]:
# "main"
conversation: list[types.Content] = []

while True:
    query = input("Pregunta a gemini, 'exit' para salir")
    if query.lower() in ("exit", "salir"):
        break
        
    response = run_llm_with_tools(
        user_input = query,
        conversation = conversation
    )

    print("Respuesta:")
    print(response["text"])

    conversation = response["conversation"]
    